<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 13 · Sequence Models and Temporal Deep Learning

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
Bring Chapter 13 sequence models to life with an LSTM and Transformer sketch for predicting next-day returns using rolling windows.

### Getting Help While Using Sequence Models
- **Appendix D** details PyTorch sequence modules.
- **Chapter 7** shows how to build rolling windows for time-series data.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

## 1. Create Rolling Sequence Dataset

We slice the AAPL return series into overlapping windows so that each example contains a look‑back history and a next‑day prediction target.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()["AAPL"]
window = 30
X_seq = []
y_seq = []
for i in range(len(log_rets) - window - 1):
    X_seq.append(log_rets.iloc[i : i + window].values)
    y_seq.append(log_rets.iloc[i + window + 1])
X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32)
X_seq = torch.from_numpy(X_seq).unsqueeze(-1)
y_seq = torch.from_numpy(y_seq)
train_len = int(len(X_seq) * 0.8)
train_loader = DataLoader(TensorDataset(X_seq[:train_len], y_seq[:train_len]),
batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(X_seq[train_len:], y_seq[train_len:]),
batch_size=256)

## 2. LSTM Model

This section introduces a small LSTM that ingests each window as a sequence and outputs a single next‑day return forecast.

In [ ]:
class ReturnLSTM(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        h, _ = self.lstm(x)
        last = h[:, -1, :]
        return self.out(last).squeeze(-1)

lstm_model = ReturnLSTM()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

### 2.1 Train LSTM

We train the LSTM for a few epochs, monitoring validation loss to gauge how well it captures temporal patterns.

In [ ]:
def train_seq(model, epochs=5):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_losses = [loss_fn(model(xb), yb).item() for xb, yb in val_loader]
        print(f"Epoch {epoch+1}, val loss {np.mean(val_losses):.6f}")

train_seq(lstm_model, epochs=5)

## 3. Transformer Encoder Sketch

Here we define a lightweight Transformer encoder to process the same sequences using self‑attention instead of recurrent state.

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, d_model=32, nhead=4, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.input_proj(x)
        encoded = self.encoder(h)
        last = encoded[:, -1, :]
        return self.out(last).squeeze(-1)

transformer = SimpleTransformer()
opt_trans = torch.optim.Adam(transformer.parameters(), lr=1e-3)

### 3.1 Train Transformer

We run a short training loop for the Transformer to compare its validation loss with the LSTM baseline.

In [ ]:
for epoch in range(3):
    transformer.train()
    for xb, yb in train_loader:
        opt_trans.zero_grad()
        loss = loss_fn(transformer(xb), yb)
        loss.backward()
        opt_trans.step()
    transformer.eval()
    with torch.no_grad():
        val_losses = [loss_fn(transformer(xb), yb).item() for xb, yb in val_loader]
    print(f"Epoch {epoch+1}, val loss {np.mean(val_losses):.6f}")

## 4. Exercises
### Exercise 1 – Sequence Length Sensitivity
Test window sizes (15, 60) and log validation loss per configuration.
<details><summary>Hint</summary>
Wrap the dataset creation in a function parameterized by <code>window</code>.
</details>

### Exercise 2 – Attention Visualization
Extract attention weights from the Transformer encoder to inspect which timesteps matter most.
<details><summary>Hint</summary>
Modify <code>SimpleTransformer</code> to return <code>attn_output_weights</code> from <code>encoder_layer</code>.
</details>

### Exercise 3 – Hybrid Signal
Combine LSTM and Transformer predictions (average) and evaluate portfolio stats.
<details><summary>Hint</summary>
Store both prediction series and feed their average into <code>performance_stats</code>.
</details>


## 5. Takeaways for Chapter 13
- Sequence windows turn single-asset series into supervised learning datasets.
- LSTMs and Transformers differ mainly in how they summarize temporal structure.
- Validation monitoring is critical given the limited sample sizes typical in finance.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">